# Ablation study — ladder and reference baselines

This notebook **runs no simulation**: it reads the artifacts of a campaign that
has already been produced.

```bash
python main.py run --config experiments/ablation.yaml
```

It answers two distinct questions, both read on the **same** world (same seed,
same grid, same fleet, same behaviour draws).

**1. Which component does the Nearest / BRAM-EV Full gap come from?**
The ladder adds one component at a time, so the gap between two consecutive
rungs *is* the contribution of the component added.

| Configuration | Method | Multi-station | Reputation | Adaptation |
| --- | --- | :-: | :-: | :-: |
| Nearest | `greedy` | no | no | no |
| Multi-Station Only | `multistation` | yes | no | no |
| Multi-Station + Reputation | `multistation_rep` | yes | yes | no |
| BRAM-EV Full | `bramev` | yes | yes | yes |

**2. Does BRAM-EV beat simple choice policies?**
The three baselines share exactly the protocol of `multistation` — broadcast to
the stations within the search radius, with no reputation and no adaptation —
and differ from it only by the **offer selection rule**. At identical
information scope, a measured gap is therefore attributable to the rule alone.

| Baseline | Method | Choice rule |
| --- | --- | --- |
| Minimum Waiting Time | `min_waiting` | lowest waiting time |
| Load-Aware | `load_aware` | least loaded upcoming station |
| Random Feasible | `random_feasible` | uniform draw among the offers received |

`greedy` also serves as the single-station baseline: it is the only method that
contacts one station only.

The plan is declared once, in `src/experiments/methods.py` — the BRAM-EV
variants are handled by `ablation_variants.ipynb`.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

import numpy as np
import pandas as pd
from IPython.display import Image, display

import src.experiments.methods as methods
from src.pipeline import ablation, figures
from src.pipeline.store import RunStore

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)

## Choosing the run

A campaign does not necessarily carry the four rungs: `latest_with_methods`
takes the most recent run that contains them all, and says what the others
contain if it finds none. To target a specific run:
`RunStore.open('../results_grid/<run>')`.

In [ ]:
for path in RunStore.list_runs('../results_grid'):
    present = sorted({row['method'] for row in RunStore(path).read_summary()})
    print(f"{path.name}\n    {', '.join(present) or 'no case'}")

In [ ]:
store = RunStore.latest_with_methods(methods.LADDER, '../results_grid')
params = store.read_params()
manifest = store.read_manifest()

print(store.root)
print(params.describe())
print(f"seed={params.seed} | commit={manifest['git_commit']} | "
      f"cases={manifest['nb_cases_done']}/{manifest['nb_cases_planned']}")

## What is actually enabled

First check, before reading any result: the flags actually applied. They travel
from the registry all the way to `summary.csv` (`src/pipeline/tables.py`), so
this table says what the campaign did — not what it was supposed to do.

A rung that does not flip exactly one component would invalidate the whole
attribution of the gains.

In [ ]:
summary = pd.read_csv(store.summary_path)

ORDER = list(methods.LADDER) + list(methods.BASELINES)
summary['method'] = pd.Categorical(summary['method'], ORDER + [
    m for m in summary['method'].unique() if m not in ORDER], ordered=True)

ladder = summary[summary['method'].isin(methods.LADDER)].copy()
baselines = summary[summary['method'].isin(methods.BASELINES)].copy()
compare = summary[summary['method'].isin(ORDER)].copy()

plan = (compare[['method', 'method_label', 'method_family', 'broadcast',
                 'reputation', 'adaptation', 'offer_choice', 'alpha_mode',
                 'reputation_scope', 'score_weighting']]
        .drop_duplicates()
        .sort_values('method')
        .set_index('method'))
plan

In [ ]:
# One single component changes from one rung to the next, and the internal
# mechanisms stay those of BRAM-EV throughout: otherwise a rung would mix two
# effects.
components = ['broadcast', 'reputation', 'adaptation']
mechanisms = ['offer_choice', 'alpha_mode', 'reputation_scope', 'score_weighting']

for before, after, label in methods.LADDER_STEPS:
    a, b = plan.loc[before], plan.loc[after]
    changed = [c for c in components if a[c] != b[c]]
    drift = [m for m in mechanisms if a[m] != b[m]]
    state = 'OK' if (len(changed) == 1 and not drift) else 'ANOMALY'
    print(f"{state:9} {before:18} -> {after:18} {label:26} "
          f"changed={changed} drifted_mechanisms={drift}")

# A baseline must differ from `multistation` only by its choice rule: same
# broadcast, same radius, no reputation and no adaptation. Otherwise the measured
# gap would no longer be attributable to the rule.
print()
if 'multistation' in plan.index:
    ref = plan.loc['multistation']
    shared = components + ['alpha_mode', 'reputation_scope', 'score_weighting']
    for name in methods.BASELINES:
        if name not in plan.index:
            continue
        b = plan.loc[name]
        drift = [c for c in shared if ref[c] != b[c]]
        state = 'OK' if not drift else 'ANOMALY'
        print(f"{state:9} {name:18} rule={b['offer_choice']:<9} "
              f"gaps_outside_the_rule={drift}")

In [ ]:
# Decomposition computed on the fly from summary.csv. The pipeline persists
# exactly the same tables (`ablation.csv`, `ablation_mean.csv`) and rewrites them
# at every case; recomputing them here makes the notebook usable on a campaign
# still running, interrupted, or predating the ablation study.
detail = pd.DataFrame(ablation.detail_rows(summary.to_dict('records')))
means = pd.DataFrame(ablation.mean_rows(detail.to_dict('records')))

print(f"{len(detail)} gaps computed over "
      f"{detail[['scenario', 'nb_cars']].drop_duplicates().shape[0]} worlds")

## The methods side by side

The four rungs of the ladder, then the three reference baselines.

In [ ]:
METRICS = ['exact_satisfaction', 'rate_abs', 'mean_service_rate',
           'slot_waste_rate', 'nb_reservations', 'mean_waiting_time_min',
           'mean_offers_per_demand', 'total_ms_mean']

levels = ladder.pivot_table(index=['scenario', 'nb_cars'], columns='method',
                            values=METRICS, observed=True)
levels

In [ ]:
# Mean over every world of the run: one row per compared method.
# The ladder first, the baselines next — the order of `ORDER`.
(compare.groupby('method', observed=True)[METRICS]
        .mean()
        .rename(index=methods.label)
        .round(4))

## Contribution of each component

`ablation_mean.csv` is written by the pipeline at every campaign
(`src/pipeline/ablation.py`). Each row is a (component, metric) pair:

* `mean_delta` / `mean_delta_pct` — the mean gap to the previous rung;
* `share_improved` — the **share of the worlds** where the component improves
  the metric, its direction being declared per metric (fewer no-shows is a
  gain, less satisfaction is not).

`share_improved` is what matters: a component that only helps half of the
worlds has no robust contribution, whatever its average.

In [ ]:
print(ablation.render_mean_table(means.to_dict('records')))

In [ ]:
ladder_means = means[means['kind'] == 'ladder']

contributions = ladder_means.pivot_table(
    index='component', columns='metric_label',
    values=['mean_delta_pct', 'share_improved'])
contributions.round(3)

### Do the contributions sum to the total gap?

The three consecutive gaps must add up to the Nearest -> BRAM-EV Full gap,
world by world. A non-zero residual would signal a missing rung or an
inconsistent `summary.csv`.

In [ ]:
steps = detail[(detail['kind'] == 'ladder') &
               (detail['metric'] == 'exact_satisfaction')]

summed = (steps.groupby(['scenario', 'nb_cars'])['delta'].sum()
               .rename('sum_of_contributions'))

endpoints = ladder.pivot_table(index=['scenario', 'nb_cars'], columns='method',
                               values='exact_satisfaction', observed=True)
span = (endpoints[methods.LADDER[-1]] - endpoints[methods.LADDER[0]]
        ).rename('total_gap')

check = pd.concat([summed, span], axis=1)
check['residual'] = (check['sum_of_contributions'] - check['total_gap']).abs()
print(f"maximal residual: {check['residual'].max():.2e}")
check.round(6)

## BRAM-EV against the reference baselines

Reading direction **opposite** to that of the ladder: the rows go
`baseline -> bramev`, so `improvement = True` means that **BRAM-EV does better
than the baseline**. That is the question asked of a baseline; the ladder, on
the other hand, asks what a component brings.

Since the three baselines have the same information scope as `multistation`,
the gap measured here comes only from the offer selection rule.

In [ ]:
baseline_means = means[means['kind'] == 'baseline']

head_to_head = baseline_means.pivot_table(
    index='component', columns='metric_label',
    values=['mean_delta_pct', 'share_improved'])
head_to_head.round(3)

In [ ]:
# Verdict world by world, ties kept apart: `improvement` is strict, so a zero
# gap counts there as a defeat. On a lightly constrained grid, two methods often
# return *exactly* the same result — reading that as a defeat of BRAM-EV would be
# wrong.
KEYS = ['exact_satisfaction', 'rate_abs', 'mean_service_rate']

duel = detail[(detail['kind'] == 'baseline') & (detail['metric'].isin(KEYS))].copy()
duel['outcome'] = np.where(duel['delta'] == 0, 'tie',
                           np.where(duel['improvement'], 'won', 'lost'))

tally = (duel.groupby(['from_method', 'metric', 'outcome'], observed=True)
             .size().unstack('outcome', fill_value=0)
             .reindex(columns=['won', 'tie', 'lost'], fill_value=0))

verdict = (tally.apply(lambda r: f"{r['won']}W / {r['tie']}T / {r['lost']}L", axis=1)
                .unstack('metric')
                .rename(index=methods.label))
print("BRAM-EV against each baseline (W won / T tie / L lost, per world):")
print(verdict, end='\n\n')

lost = tally[tally['lost'] > tally['won']]
if len(lost):
    for (m, metric), r in lost.iterrows():
        print(f"WARNING: against {methods.label(m)}, BRAM-EV loses on "
              f"{metric} ({r['lost']}L against {r['won']}W).")
elif (tally['won'] == 0).all():
    print("No gap at all: the methods return the same result on every world. "
          "The grid is not constrained enough to separate them — lower "
          "nb_stations or raise the fleet.")
else:
    print("BRAM-EV is beaten by no baseline on these metrics.")

In [ ]:
# Raw levels, without going through the gaps: what each choice rule actually
# produces. `min_waiting` should dominate the waiting time, `random_feasible`
# is the floor.
COLUMNS = ['exact_satisfaction', 'mean_waiting_time_min',
           'mean_travel_distance_km', 'rate_abs', 'mean_service_rate']

(compare.groupby('method', observed=True)[COLUMNS]
        .mean()
        .rename(index=methods.label)
        .round(4))

## Dispersion: what does the average hide?

A positive mean gap may be carried by a single world. Here is the distribution
of the contributions per component, over every fleet size and every scenario.

In [ ]:
for metric in ['exact_satisfaction', 'rate_abs', 'mean_service_rate']:
    block = detail[(detail['kind'] == 'ladder') & (detail['metric'] == metric)]
    if block.empty:
        continue
    label = block['metric_label'].iloc[0]
    print(f"\n=== {label} — gap to the previous rung ===")
    stats = (block.groupby('component')['delta']
                  .agg(['count', 'min', 'median', 'mean', 'max'])
                  .round(6))
    stats['worlds_improved'] = block.groupby('component')['improvement'].mean().round(3)
    display(stats)

In [ ]:
# Does the component help more as the resource becomes scarce?
# A contribution that grows with the fleet is a contribution that bears on
# congestion, not on the luck of the draw.
(detail[(detail['kind'] == 'ladder') &
        (detail['metric'] == 'exact_satisfaction')]
 .pivot_table(index='component', columns='nb_cars', values='delta_pct')
 .round(2))

## Figures

Already written by the pipeline in `figures/`. To regenerate them after editing
`src/pipeline/figures.py`:

```bash
python main.py report --latest
```

In [ ]:
for name in ['ablation_components', 'scalability']:
    path = store.figure_path(name)
    if path.is_file():
        print(path.name)
        display(Image(filename=str(path)))

for path in sorted(store.figures_dir.glob('ablation_ladder_*.png')):
    print(path.name)
    display(Image(filename=str(path)))

In [ ]:
# Or rebuild a figure in memory, without going through the disk.
figures.fig_ablation_components(summary.to_dict('records'))

## Reading guards

Two components can structurally show nothing on a campaign that is too short or
too lightly constrained. A `+0.0%` must then be read as "mechanism never
solicited", not "useless component".

1. **Cross-station adaptation** only fires every `SOCIETY_UPDATE_INTERVAL`
   slots (24 by default, i.e. 2 h). The cell below computes how many times it
   actually fired.
2. **Multi-station search** only has an effect if the request does reach more
   than one station, and if capacity is constrained: with free chargers
   everywhere, every demand is served anyway.

In [ ]:
config = next(store.iter_results())['config']
interval = config['society_update_interval']
nb_updates = params.total_time // interval

print(f"horizon = {params.total_time} slots | "
      f"adaptation interval = {interval} slots")
print(f"-> collective learning fires {nb_updates} time(s)")
if nb_updates == 0:
    print("   WARNING: never fired — the measured contribution of the "
          "adaptation is zero by construction.")

In [ ]:
broadcast_stats = (ladder.groupby('method', observed=True)
                   .agg(offers_per_demand=('mean_offers_per_demand', 'mean'),
                        service_rate=('mean_service_rate', 'mean'),
                        rejected_demands=('nb_station_level_rejections', 'mean'))
                   .round(3))
print(broadcast_stats, end='\n\n')

offer_gain = (broadcast_stats.loc['multistation', 'offers_per_demand']
              - broadcast_stats.loc['greedy', 'offers_per_demand'])
if offer_gain <= 0:
    print("WARNING: broadcasting does not produce more offers per demand. "
          "The search radius is too small against the spacing of the "
          "stations: the first rung is not being tested.")
else:
    print(f"Broadcasting brings {offer_gain:+.2f} offer(s) per demand.")

## Health of the run

A violated invariant or a massive diagnostic makes every conclusion suspect:
to be checked before quoting any figure.

In [ ]:
print('invariants OK:', bool(summary['invariant_ok'].all()))
print('unresolved reservations:', int(summary['nb_unresolved'].sum()))
print('breakdowns:', int(summary['nb_breakdowns'].sum()))

seen = set()
for result in store.iter_results():
    for message in result['behaviors'].get('diagnostics', []):
        if message not in seen:
            seen.add(message)
            print(f"\n[diagnostic] {message}")